In [ ]:
# !pip install yfinance vectorbt TA-Lib matplotlib scipy pandas numpy

In [ ]:
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys
import os
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import combinations

warnings.filterwarnings('ignore')

# Add lib to path
sys.path.insert(0, os.path.join(os.getcwd(), 'lib'))
from lib.data_manager import download

print('Imports loaded.')

# Multi-Signal Ensemble Builder

- Tests all 2-signal and 3-signal combinations from a pool of indicators
- Entry requires ALL signals to agree (AND logic) or ANY signal (OR logic)
- Each ensemble gets IS grid search + OOS validation
- Boruta validation: shuffle OOS returns 100+ times, check if real Sharpe beats noise
- Designed for daily and intraday timeframes

In [ ]:
# CONFIGURATION

TICKER = 'BTC-USD'
START_DATE = '2018-01-01'
TIMEFRAME = 'D'  # 'D', '1h', '4h'
TRAIN_RATIO = 0.60
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005

# Ensemble logic: 'AND' (all must agree) or 'OR' (any triggers)
ENSEMBLE_LOGIC = 'AND'

# Boruta settings
N_BORUTA_SHUFFLES = 100
BORUTA_THRESHOLD = 0.80  # 80% = statistically significant

# Signal pool — each generates a boolean entry/exit pair
# Parameters here are the "base" values; can be grid-searched later
SIGNAL_POOL = {
    'macd_fast': {'fast': 12, 'slow': 26, 'signal': 9},
    'macd_slow': {'fast': 20, 'slow': 50, 'signal': 12},
    'rsi_oversold': {'rsi_len': 14, 'oversold': 30, 'overbought': 70},
    'rsi_deep': {'rsi_len': 14, 'oversold': 20, 'overbought': 80},
    'donchian_short': {'entry': 10, 'exit': 5, 'filter': 20},
    'donchian_long': {'entry': 30, 'exit': 15, 'filter': 50},
    'ema_cross_fast': {'fast_ema': 8, 'slow_ema': 21},
    'ema_cross_slow': {'fast_ema': 13, 'slow_ema': 50},
    'stoch_oversold': {'k_period': 14, 'd_period': 3, 'oversold': 20, 'overbought': 80},
    'bollinger_squeeze': {'bb_period': 20, 'bb_std': 2.0},
    'atr_breakout': {'atr_period': 14, 'atr_mult': 1.5},
    'sma_trend': {'sma_period': 50},
}

print(f"Signal pool: {len(SIGNAL_POOL)} indicators")
from itertools import combinations
n2 = len(list(combinations(SIGNAL_POOL.keys(), 2)))
n3 = len(list(combinations(SIGNAL_POOL.keys(), 3)))
print(f"2-signal combos: {n2}")
print(f"3-signal combos: {n3}")
print(f"Total ensembles to test: {n2 + n3}")

In [ ]:
# DOWNLOAD DATA

print(f"Downloading {TICKER} from {START_DATE}...")
df = download(TICKER, START_DATE, timeframe=TIMEFRAME)

# Flatten MultiIndex columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]

close = df['Close'].squeeze()
high = df['High'].squeeze()
low = df['Low'].squeeze()

print(f"\n{len(df)} bars loaded: {df.index[0].date()} to {df.index[-1].date()}")
print(df.tail(3))

In [ ]:
# INDIVIDUAL SIGNAL GENERATORS
# Each returns (entries: pd.Series[bool], exits: pd.Series[bool])
# All apply 1-bar execution delay

def sig_macd(close, fast, slow, signal):
    ml, sl, _ = talib.MACD(close.values.astype(float), fastperiod=fast, slowperiod=slow, signalperiod=signal)
    macd_s = pd.Series(ml, index=close.index)
    sig_s = pd.Series(sl, index=close.index)
    entries = (macd_s.shift(1) <= sig_s.shift(1)) & (macd_s > sig_s)
    exits = (macd_s.shift(1) >= sig_s.shift(1)) & (macd_s < sig_s)
    return entries.shift(1).fillna(False).astype(bool), exits.shift(1).fillna(False).astype(bool)

def sig_rsi(close, rsi_len, oversold, overbought):
    rsi = pd.Series(talib.RSI(close.values, timeperiod=rsi_len), index=close.index)
    entries = (rsi.shift(1) <= oversold) & (rsi > oversold)
    exits = (rsi.shift(1) <= overbought) & (rsi > overbought)
    return entries.shift(1).fillna(False).astype(bool), exits.shift(1).fillna(False).astype(bool)

def sig_donchian(close, high, low, entry_period, exit_period, filter_period):
    upper = pd.Series(talib.MAX(high.values, entry_period), index=close.index).shift(1)
    lower = pd.Series(talib.MIN(low.values, exit_period), index=close.index).shift(1)
    trend = pd.Series(talib.SMA(close.values, filter_period), index=close.index).shift(1)
    entries = (close > upper) & (close > trend)
    exits = (close < lower)
    return entries.shift(1).fillna(False).astype(bool), exits.shift(1).fillna(False).astype(bool)

def sig_ema_cross(close, fast_ema, slow_ema):
    fast = pd.Series(talib.EMA(close.values, fast_ema), index=close.index)
    slow = pd.Series(talib.EMA(close.values, slow_ema), index=close.index)
    entries = (fast.shift(1) <= slow.shift(1)) & (fast > slow)
    exits = (fast.shift(1) >= slow.shift(1)) & (fast < slow)
    return entries.shift(1).fillna(False).astype(bool), exits.shift(1).fillna(False).astype(bool)

def sig_stochastic(close, high, low, k_period, d_period, oversold, overbought):
    slowk, slowd = talib.STOCH(high.values, low.values, close.values,
                                fastk_period=k_period, slowk_period=d_period,
                                slowk_matype=0, slowd_period=d_period, slowd_matype=0)
    k = pd.Series(slowk, index=close.index)
    entries = (k.shift(1) <= oversold) & (k > oversold)
    exits = (k.shift(1) >= overbought) & (k > overbought)
    return entries.shift(1).fillna(False).astype(bool), exits.shift(1).fillna(False).astype(bool)

def sig_bollinger(close, bb_period, bb_std):
    upper, middle, lower = talib.BBANDS(close.values, timeperiod=bb_period, nbdevup=bb_std, nbdevdn=bb_std)
    upper_s = pd.Series(upper, index=close.index)
    lower_s = pd.Series(lower, index=close.index)
    # Mean reversion: enter when price touches lower band, exit at middle
    entries = (close.shift(1) > lower_s.shift(1)) & (close <= lower_s)  # crosses below lower
    exits = (close > pd.Series(middle, index=close.index))
    return entries.shift(1).fillna(False).astype(bool), exits.shift(1).fillna(False).astype(bool)

def sig_atr_breakout(close, high, low, atr_period, atr_mult):
    atr = pd.Series(talib.ATR(high.values, low.values, close.values, atr_period), index=close.index)
    sma = pd.Series(talib.SMA(close.values, atr_period), index=close.index)
    upper_band = sma + atr * atr_mult
    entries = (close.shift(1) <= upper_band.shift(1)) & (close > upper_band)
    exits = close < sma
    return entries.shift(1).fillna(False).astype(bool), exits.shift(1).fillna(False).astype(bool)

def sig_sma_trend(close, sma_period):
    sma = pd.Series(talib.SMA(close.values, sma_period), index=close.index)
    # Trend filter: entry when price crosses above SMA, exit when crosses below
    entries = (close.shift(1) <= sma.shift(1)) & (close > sma)
    exits = (close.shift(1) >= sma.shift(1)) & (close < sma)
    return entries.shift(1).fillna(False).astype(bool), exits.shift(1).fillna(False).astype(bool)


def generate_signal(name, close, high, low, params):
    """Dispatch to the appropriate signal generator."""
    if name.startswith('macd'):
        return sig_macd(close, params['fast'], params['slow'], params['signal'])
    elif name.startswith('rsi'):
        return sig_rsi(close, params['rsi_len'], params['oversold'], params['overbought'])
    elif name.startswith('donchian'):
        return sig_donchian(close, high, low, params['entry'], params['exit'], params['filter'])
    elif name.startswith('ema_cross'):
        return sig_ema_cross(close, params['fast_ema'], params['slow_ema'])
    elif name.startswith('stoch'):
        return sig_stochastic(close, high, low, params['k_period'], params['d_period'],
                              params['oversold'], params['overbought'])
    elif name.startswith('bollinger'):
        return sig_bollinger(close, params['bb_period'], params['bb_std'])
    elif name.startswith('atr'):
        return sig_atr_breakout(close, high, low, params['atr_period'], params['atr_mult'])
    elif name.startswith('sma'):
        return sig_sma_trend(close, params['sma_period'])
    else:
        raise ValueError(f"Unknown signal: {name}")

print(f"{len(SIGNAL_POOL)} signal generators defined.")

In [ ]:
# PRE-COMPUTE ALL INDIVIDUAL SIGNALS

print(f"Pre-computing {len(SIGNAL_POOL)} individual signals for {TICKER}...")

individual_signals = {}
for name, params in SIGNAL_POOL.items():
    try:
        entries, exits = generate_signal(name, close, high, low, params)
        n_entries = entries.sum()
        individual_signals[name] = {'entries': entries, 'exits': exits, 'n_entries': n_entries}
        print(f"  {name:20s}: {n_entries:>5} entry signals")
    except Exception as e:
        print(f"  {name:20s}: FAILED \u2014 {e}")

print(f"\n{len(individual_signals)} signals computed successfully.")

In [ ]:
# BUILD & TEST ALL ENSEMBLES (2-signal and 3-signal)

from itertools import combinations
import time

split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx]
val_close = close.iloc[split_idx:]

signal_names = list(individual_signals.keys())
all_combos = []
for r in [2, 3]:
    all_combos.extend(list(combinations(signal_names, r)))

print(f"Testing {len(all_combos)} ensembles ({ENSEMBLE_LOGIC} logic)...")
t0 = time.time()

ensemble_results = []

for idx, combo in enumerate(all_combos):
    # Combine signals
    entry_signals = [individual_signals[name]['entries'] for name in combo]
    exit_signals = [individual_signals[name]['exits'] for name in combo]
    
    if ENSEMBLE_LOGIC == 'AND':
        combined_entries = entry_signals[0].copy()
        for s in entry_signals[1:]:
            combined_entries = combined_entries & s
        # Exit on ANY exit signal
        combined_exits = exit_signals[0].copy()
        for s in exit_signals[1:]:
            combined_exits = combined_exits | s
    elif ENSEMBLE_LOGIC == 'OR':
        combined_entries = entry_signals[0].copy()
        for s in entry_signals[1:]:
            combined_entries = combined_entries | s
        # Exit when ALL exits agree
        combined_exits = exit_signals[0].copy()
        for s in exit_signals[1:]:
            combined_exits = combined_exits & s
    
    # IS backtest
    train_entries = combined_entries.iloc[:split_idx]
    train_exits = combined_exits.iloc[:split_idx]
    
    try:
        pf_is = vbt.Portfolio.from_signals(
            close=train_close, entries=train_entries, exits=train_exits,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME
        )
        
        n_trades = len(pf_is.trades)
        if n_trades < 10:
            continue
        
        is_sharpe = float(pf_is.sharpe_ratio(freq=TIMEFRAME))
        if np.isnan(is_sharpe) or np.isinf(is_sharpe):
            continue
        
        # OOS backtest
        val_entries = combined_entries.iloc[split_idx:]
        val_exits = combined_exits.iloc[split_idx:]
        
        pf_oos = vbt.Portfolio.from_signals(
            close=val_close, entries=val_entries, exits=val_exits,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME
        )
        
        oos_sharpe = float(pf_oos.sharpe_ratio(freq=TIMEFRAME))
        oos_trades = len(pf_oos.trades)
        
        tr = pf_oos.trades.returns.values if hasattr(pf_oos.trades.returns, 'values') else np.array(pf_oos.trades.returns)
        win_rate = (tr > 0).sum() / len(tr) * 100 if len(tr) > 0 else 0
        
        ensemble_results.append({
            'ensemble': ' + '.join(combo),
            'n_signals': len(combo),
            'logic': ENSEMBLE_LOGIC,
            'is_sharpe': is_sharpe,
            'oos_sharpe': oos_sharpe,
            'sharpe_degradation': (is_sharpe - oos_sharpe) / abs(is_sharpe) if abs(is_sharpe) > 0.01 else 0,
            'is_return': float(pf_is.total_return()),
            'oos_return': float(pf_oos.total_return()),
            'is_maxdd': float(pf_is.max_drawdown()),
            'oos_maxdd': float(pf_oos.max_drawdown()),
            'is_trades': n_trades,
            'oos_trades': oos_trades,
            'win_rate': win_rate,
        })
    except Exception:
        continue
    
    if (idx + 1) % 50 == 0:
        print(f"  {idx+1}/{len(all_combos)} ensembles tested | Valid: {len(ensemble_results)}")

elapsed = time.time() - t0
ens_df = pd.DataFrame(ensemble_results).sort_values('oos_sharpe', ascending=False)

print(f"\nDone in {elapsed:.1f}s \u2014 {len(ens_df)} valid ensembles")
print(f"\nTop 15 by OOS Sharpe:")
print(ens_df.head(15)[['ensemble', 'n_signals', 'is_sharpe', 'oos_sharpe', 
                         'sharpe_degradation', 'oos_return', 'oos_maxdd', 'oos_trades', 'win_rate']].to_string(index=False))

In [ ]:
# BORUTA VALIDATION \u2014 STATISTICAL SIGNIFICANCE TEST

print("=" * 70)
print(f"BORUTA VALIDATION \u2014 {N_BORUTA_SHUFFLES} shuffles per ensemble")
print("=" * 70)

# Take top 25 ensembles by IS Sharpe for Boruta validation
top_ensembles = ens_df.sort_values('is_sharpe', ascending=False).head(25)

boruta_results = []

for idx, row in top_ensembles.iterrows():
    combo_names = row['ensemble'].split(' + ')
    
    # Get OOS signals
    entry_signals = [individual_signals[name]['entries'].iloc[split_idx:] for name in combo_names]
    exit_signals = [individual_signals[name]['exits'].iloc[split_idx:] for name in combo_names]
    
    if ENSEMBLE_LOGIC == 'AND':
        combined_entries = entry_signals[0].copy()
        for s in entry_signals[1:]:
            combined_entries = combined_entries & s
        combined_exits = exit_signals[0].copy()
        for s in exit_signals[1:]:
            combined_exits = combined_exits | s
    else:
        combined_entries = entry_signals[0].copy()
        for s in entry_signals[1:]:
            combined_entries = combined_entries | s
        combined_exits = exit_signals[0].copy()
        for s in exit_signals[1:]:
            combined_exits = combined_exits & s
    
    # Real OOS portfolio
    try:
        pf_real = vbt.Portfolio.from_signals(
            close=val_close, entries=combined_entries, exits=combined_exits,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME
        )
        real_sharpe = float(pf_real.sharpe_ratio(freq=TIMEFRAME))
        real_returns = pf_real.returns().values.ravel()
    except Exception:
        continue
    
    # Shuffle test
    beats_shadow = 0
    for shuffle_i in range(N_BORUTA_SHUFFLES):
        shuffled_rets = np.random.permutation(real_returns)
        shuffled_eq = INIT_CASH * np.cumprod(1 + shuffled_rets)
        shadow_total_ret = shuffled_eq[-1] / INIT_CASH - 1
        n_years = len(shuffled_rets) / 252
        shadow_ann_ret = (1 + shadow_total_ret) ** (1 / n_years) - 1 if n_years > 0 else 0
        shadow_vol = np.std(shuffled_rets) * np.sqrt(252)
        shadow_sharpe = shadow_ann_ret / shadow_vol if shadow_vol > 0 else 0
        
        if real_sharpe > shadow_sharpe:
            beats_shadow += 1
    
    boruta_score = beats_shadow / N_BORUTA_SHUFFLES
    verdict = "CONFIRMED" if boruta_score >= BORUTA_THRESHOLD else "REJECTED"
    
    boruta_results.append({
        'ensemble': row['ensemble'],
        'n_signals': row['n_signals'],
        'is_sharpe': row['is_sharpe'],
        'oos_sharpe': real_sharpe,
        'boruta_score': boruta_score,
        'boruta_verdict': verdict,
        'oos_return': float(pf_real.total_return()),
        'oos_maxdd': float(pf_real.max_drawdown()),
        'oos_trades': len(pf_real.trades),
    })
    
    flag = "CONFIRMED" if boruta_score >= BORUTA_THRESHOLD else "rejected"
    print(f"  {row['ensemble'][:50]:50s} | Boruta={boruta_score:.0%} {flag} | OOS Sharpe={real_sharpe:.3f}")

boruta_df = pd.DataFrame(boruta_results).sort_values('boruta_score', ascending=False)
confirmed = boruta_df[boruta_df['boruta_verdict'] == 'CONFIRMED']

print(f"\n{'='*70}")
print(f"BORUTA RESULTS: {len(confirmed)}/{len(boruta_df)} ensembles CONFIRMED (score >= {BORUTA_THRESHOLD:.0%})")
print(f"{'='*70}")
if not confirmed.empty:
    print(confirmed.to_string(index=False))

In [ ]:
# MONTE CARLO FTMO \u2014 BORUTA-CONFIRMED ENSEMBLES

FTMO_ACCOUNT = 100_000
PROFIT_TARGET = 0.10
MAX_DAILY_LOSS = 0.05
MAX_TOTAL_LOSS = 0.10
TRADING_DAYS = 30
N_SIMS = 10_000

mc_results = []
target = confirmed if not confirmed.empty else boruta_df.head(5)

for _, row in target.iterrows():
    combo_names = row['ensemble'].split(' + ')
    
    entry_signals = [individual_signals[name]['entries'] for name in combo_names]
    exit_signals = [individual_signals[name]['exits'] for name in combo_names]
    
    if ENSEMBLE_LOGIC == 'AND':
        combined_entries = entry_signals[0].copy()
        for s in entry_signals[1:]:
            combined_entries = combined_entries & s
        combined_exits = exit_signals[0].copy()
        for s in exit_signals[1:]:
            combined_exits = combined_exits | s
    else:
        combined_entries = entry_signals[0].copy()
        for s in entry_signals[1:]:
            combined_entries = combined_entries | s
        combined_exits = exit_signals[0].copy()
        for s in exit_signals[1:]:
            combined_exits = combined_exits & s
    
    pf = vbt.Portfolio.from_signals(
        close=close, entries=combined_entries, exits=combined_exits,
        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME
    )
    
    daily_rets = pf.returns().values.ravel()
    daily_rets = daily_rets[~np.isnan(daily_rets)]
    
    np.random.seed(42)
    n_passed = n_blown_total = n_blown_daily = 0
    finals = []
    
    for _ in range(N_SIMS):
        sim_rets = np.random.choice(daily_rets, size=TRADING_DAYS, replace=True)
        eq = FTMO_ACCOUNT
        for r in sim_rets:
            day_start = eq
            eq *= (1 + r)
            if (eq - day_start) / FTMO_ACCOUNT < -MAX_DAILY_LOSS:
                n_blown_daily += 1; break
            if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT < -MAX_TOTAL_LOSS:
                n_blown_total += 1; break
            if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT >= PROFIT_TARGET:
                n_passed += 1; break
        finals.append(eq)
    
    pass_rate = n_passed / N_SIMS * 100
    verdict = "FAVORABLE" if pass_rate >= 50 else "POSSIBLE" if pass_rate >= 25 else "CHALLENGING" if pass_rate >= 10 else "UNLIKELY"
    
    mc_results.append({
        'ensemble': row['ensemble'], 'boruta_score': row['boruta_score'],
        'oos_sharpe': row['oos_sharpe'], 'pass_rate': pass_rate, 'verdict': verdict,
        'median_equity': np.median(finals),
    })
    print(f"  {row['ensemble'][:45]:45s} | Boruta={row['boruta_score']:.0%} | Pass={pass_rate:.1f}% ({verdict})")

mc_ens_df = pd.DataFrame(mc_results)
print(f"\n{'='*60}")
print(mc_ens_df.sort_values('pass_rate', ascending=False).to_string(index=False))

In [ ]:
# VISUALIZATION

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- 1. IS vs OOS Sharpe scatter, colored by Boruta score ---
ax = axes[0]
if not boruta_df.empty:
    sc = ax.scatter(boruta_df['is_sharpe'], boruta_df['oos_sharpe'],
                    c=boruta_df['boruta_score'], cmap='RdYlGn', s=80, edgecolors='k', linewidths=0.5,
                    vmin=0, vmax=1)
    plt.colorbar(sc, ax=ax, label='Boruta Score')
    # Identity line
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='IS = OOS')
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('IS Sharpe')
ax.set_ylabel('OOS Sharpe')
ax.set_title('IS vs OOS Sharpe (Boruta Top 25)')
ax.legend()

# --- 2. Ensemble comparison bar chart (top 10 by OOS Sharpe) ---
ax = axes[1]
top10 = ens_df.head(10)
if not top10.empty:
    y_pos = range(len(top10))
    labels = [e[:30] for e in top10['ensemble']]
    ax.barh(y_pos, top10['oos_sharpe'], color='steelblue', edgecolor='k', alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('OOS Sharpe Ratio')
    ax.set_title('Top 10 Ensembles by OOS Sharpe')
    ax.invert_yaxis()

# --- 3. Top ensemble equity curves (IS + OOS) ---
ax = axes[2]
if not ens_df.empty:
    top3 = ens_df.head(3)
    for _, row in top3.iterrows():
        combo_names = row['ensemble'].split(' + ')
        entry_signals = [individual_signals[name]['entries'] for name in combo_names]
        exit_signals = [individual_signals[name]['exits'] for name in combo_names]
        
        if ENSEMBLE_LOGIC == 'AND':
            ce = entry_signals[0].copy()
            for s in entry_signals[1:]:
                ce = ce & s
            cx = exit_signals[0].copy()
            for s in exit_signals[1:]:
                cx = cx | s
        else:
            ce = entry_signals[0].copy()
            for s in entry_signals[1:]:
                ce = ce | s
            cx = exit_signals[0].copy()
            for s in exit_signals[1:]:
                cx = cx & s
        
        pf = vbt.Portfolio.from_signals(
            close=close, entries=ce, exits=cx,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME
        )
        eq = pf.value()
        label = row['ensemble'][:35]
        ax.plot(eq.index, eq.values, label=label, linewidth=1.2)
    
    # Mark train/val split
    split_date = close.index[split_idx]
    ax.axvline(split_date, color='red', linestyle='--', alpha=0.7, label='IS/OOS Split')
    ax.set_xlabel('Date')
    ax.set_ylabel('Portfolio Value ($)')
    ax.set_title('Top 3 Ensemble Equity Curves')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Visualization complete.")

In [ ]:
# EXPORT + TELEGRAM NOTIFICATION

import json
from datetime import datetime

STRATEGY_NAME = 'Multi_Signal_Ensemble'

# Determine export path (Google Drive on Colab, local otherwise)
if os.path.exists('/content/drive/MyDrive'):
    export_base = f'/content/drive/MyDrive/QS_Finance/strategy_exports/{STRATEGY_NAME}/{TICKER}'
else:
    export_base = os.path.join(os.getcwd(), 'exports', STRATEGY_NAME, TICKER)

os.makedirs(os.path.join(export_base, 'latest'), exist_ok=True)
os.makedirs(os.path.join(export_base, 'archive'), exist_ok=True)

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')

# --- Summary JSON ---
best = ens_df.iloc[0] if not ens_df.empty else {}
summary = {
    'strategy': STRATEGY_NAME,
    'ticker': TICKER,
    'timeframe': TIMEFRAME,
    'start_date': START_DATE,
    'ensemble_logic': ENSEMBLE_LOGIC,
    'n_signals_in_pool': len(SIGNAL_POOL),
    'total_ensembles_tested': len(ens_df),
    'boruta_confirmed': len(confirmed) if not confirmed.empty else 0,
    'boruta_threshold': BORUTA_THRESHOLD,
    'n_boruta_shuffles': N_BORUTA_SHUFFLES,
    'train_ratio': TRAIN_RATIO,
    'init_cash': INIT_CASH,
    'fees': FEES,
    'slippage': SLIPPAGE,
    'best_ensemble': best.get('ensemble', 'N/A') if isinstance(best, dict) else best['ensemble'],
    'best_oos_sharpe': float(best.get('oos_sharpe', 0)) if isinstance(best, dict) else float(best['oos_sharpe']),
    'run_id': run_id,
    'timestamp': datetime.now().isoformat(),
}

summary_path = os.path.join(export_base, 'latest', 'summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary: {summary_path}")

# Archive copy
archive_path = os.path.join(export_base, 'archive', f'{run_id}_summary.json')
with open(archive_path, 'w') as f:
    json.dump(summary, f, indent=2)

# --- OOS ensemble results CSV ---
if not ens_df.empty:
    oos_path = os.path.join(export_base, 'latest', 'ensemble_results.csv')
    ens_df.to_csv(oos_path, index=False)
    print(f"Ensemble results: {oos_path}")

# --- Boruta results CSV ---
if not boruta_df.empty:
    boruta_path = os.path.join(export_base, 'latest', 'boruta_results.csv')
    boruta_df.to_csv(boruta_path, index=False)
    print(f"Boruta results: {boruta_path}")

# --- Monte Carlo results CSV ---
if mc_results:
    mc_path = os.path.join(export_base, 'latest', 'monte_carlo_ftmo.csv')
    mc_ens_df.to_csv(mc_path, index=False)
    print(f"Monte Carlo: {mc_path}")

# --- Telegram notification ---
try:
    import requests as _req
    
    # Try loading Telegram creds from Colab secrets or env
    try:
        from google.colab import userdata
        TG_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
        TG_CHAT = userdata.get('TELEGRAM_CHAT_ID')
    except Exception:
        TG_TOKEN = os.environ.get('TELEGRAM_BOT_TOKEN')
        TG_CHAT = os.environ.get('TELEGRAM_CHAT_ID')
    
    if TG_TOKEN and TG_CHAT:
        n_confirmed = len(confirmed) if not confirmed.empty else 0
        top_ens = confirmed.head(3) if not confirmed.empty else boruta_df.head(3)
        
        lines = [f"Multi-Signal Ensemble Builder"]
        lines.append(f"{TICKER} | {ENSEMBLE_LOGIC} logic | {len(ens_df)} ensembles tested")
        lines.append(f"Boruta confirmed: {n_confirmed}/{len(boruta_df)}")
        lines.append("")
        for _, r in top_ens.iterrows():
            lines.append(f"{r['ensemble'][:40]}")
            lines.append(f"  OOS Sharpe={r['oos_sharpe']:.3f} | Boruta={r['boruta_score']:.0%}")
        
        msg = '\n'.join(lines)
        _req.post(
            f'https://api.telegram.org/bot{TG_TOKEN}/sendMessage',
            json={'chat_id': TG_CHAT, 'text': msg}
        )
        print(f"Telegram notification sent.")
    else:
        print("Telegram credentials not found \u2014 skipping notification.")
except Exception as e:
    print(f"Telegram notification failed: {e}")

print(f"\nExport complete. Run ID: {run_id}")